# Masked Multi-Task GNN

## Scientific objective
Train a shared molecular graph encoder with endpoint-specific heads and masked endpoint losses.

## Inputs
- Global molecule/scaffold split
- Wide endpoint label matrix

## Expected outputs
- `models/multitask/gnn_multitask.pt`
- `results/metrics/multitask_gnn.csv`
- transfer diagnostics

## Dependencies
PyTorch Geometric

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
Missing labels remain masked. Shared graph features do not imply shared toxicological mechanisms.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Graph multi-task training is computationally heavier and may underperform simple QSAR in low-data/scaffold-novel regimes.

## Next notebook
[15_hyperparameter_optimization.ipynb](./15_hyperparameter_optimization.ipynb)


In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})


{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [2]:
from toxicity_screening.utils import atomic_write_json
try:
    import torch
    from torch_geometric.loader import DataLoader as GraphDataLoader
    from toxicity_screening.datasets import MaskedMolecularGraphDataset
    from toxicity_screening.gnn_models import MultiTaskGraphClassifier
    from toxicity_screening.graph_features import ATOM_FEATURE_DIM
    from toxicity_screening.training import train_graph_multitask_model, resolve_device
    from toxicity_screening.losses import positive_class_weights
    from toxicity_screening.metrics import binary_metrics
except ImportError as exc:
    atomic_write_json(
        {
            "status": "dependency_unavailable",
            "error": str(exc),
        },
        ROOT / "reports/multitask_gnn_dependency_status.json",
    )
    raise

endpoints = list(CONFIGS["endpoints"]["endpoints"])
records = pd.read_parquet(
    ROOT / "data/processed/modeling_records.parquet"
)

wide = (
    records.pivot_table(
        index=[
            "molecule_id",
            "standardized_smiles",
            "scaffold_split",
        ],
        columns="endpoint",
        values="label",
        aggfunc="first",
    )
    .reset_index()
)
parts = {
    p: wide[wide.scaffold_split == p]
    for p in ["train", "validation", "test"]
}
cap = PROFILE_CONFIG["sample_cap_per_endpoint"]
train = parts["train"].sample(
    min(len(parts["train"]), cap or len(parts["train"])),
    random_state=SEED,
)
loaders = {
    p: GraphDataLoader(
        MaskedMolecularGraphDataset(
            part.standardized_smiles,
            part[endpoints].to_numpy(float),
        ),
        batch_size=min(
            64,
            CONFIGS["training_config"]["batch_size"],
        ),
        shuffle=p == "train",
    )
    for p, part in {
        "train": train,
        "validation": parts["validation"],
    }.items()
}
y = torch.tensor(
    np.nan_to_num(
        train[endpoints].to_numpy(float),
        nan=0.0,
    )
)
mask = torch.tensor(
    ~np.isnan(train[endpoints].to_numpy(float)),
    dtype=torch.float32,
)
pw = positive_class_weights(y, mask)
tw = 1 / torch.sqrt(mask.sum(0).clamp_min(1))
tw = tw / tw.mean()
cfg = CONFIGS["model_config"]["neural"]["multitask_gnn"]
model = MultiTaskGraphClassifier(
    ATOM_FEATURE_DIM,
    len(endpoints),
    cfg["hidden_dim"],
    cfg["layers"],
    cfg["head_dim"],
    cfg["dropout"],
    cfg["architecture"],
)
result = train_graph_multitask_model(
    model,
    loaders["train"],
    loaders["validation"],
    epochs=PROFILE_CONFIG["max_epochs"],
    patience=PROFILE_CONFIG["patience"],
    learning_rate=CONFIGS["training_config"]["optimizer"]["learning_rate"],
    weight_decay=CONFIGS["training_config"]["optimizer"]["weight_decay"],
    gradient_clip_norm=CONFIGS["training_config"]["gradient_clip_norm"],
    positive_weights=pw,
    task_weights=tw,
    checkpoint_path=ROOT / "models/multitask/gnn_multitask.pt",
)
pd.DataFrame(result.history).to_csv(
    ROOT / "results/metrics/multitask_gnn_history.csv",
    index=False,
)
test_loader = GraphDataLoader(
    MaskedMolecularGraphDataset(
        parts["test"].standardized_smiles,
        parts["test"][endpoints].to_numpy(float),
    ),
    batch_size=64,
)
model.eval()
device = resolve_device()
model.to(device)
probs = []
with torch.no_grad():
    for batch in test_loader:
        probs.append(
            torch.sigmoid(
                model(batch.to(device))
            ).cpu().numpy()
        )
probs = np.vstack(probs)
ytest = parts["test"][endpoints].to_numpy(float)
rows = []
for k, e in enumerate(endpoints):
    obs = ~np.isnan(ytest[:, k])
    rows.append(
        {
            "endpoint": e,
            "model": "multitask_gnn",
            **{
                a: b
                for a, b in binary_metrics(
                    ytest[obs, k].astype(int),
                    probs[obs, k],
                ).items()
                if a != "confusion_matrix"
            },
        }
    )
pd.DataFrame(rows).to_csv(
    ROOT / "results/metrics/multitask_gnn.csv",
    index=False,
)
display(pd.DataFrame(rows))


,endpoint,model,n,positive_prevalence,threshold,roc_auc,pr_auc,mcc,accuracy,balanced_accuracy,...,f1,brier,ece,nll,recall_at_precision_0.80,precision_at_recall_0.80,tn,fp,fn,tp
0,herg_blockade,multitask_gnn,1883,0.507169,0.5,0.682101,0.686884,0.264923,0.628784,0.626720,...,0.678029,0.227426,0.054969,0.647251,0.211518,0.589961,448,480,219,736
1,ames_mutagenicity,multitask_gnn,1120,0.547321,0.5,0.732898,0.755102,0.325425,0.668750,0.657144,...,0.720422,0.207180,0.032650,0.600496,0.358891,0.658793,271,236,135,478
2,SR-p53,multitask_gnn,986,0.085193,0.5,0.814434,0.315764,0.237260,0.552738,0.712359,...,0.256324,0.224912,0.343063,0.617072,0.000000,0.191549,469,433,8,76
3,SR-ATAD5,multitask_gnn,1027,0.064265,0.5,0.864551,0.242118,0.251566,0.583252,0.756149,...,0.227437,0.228959,0.359467,0.629558,0.000000,0.193220,536,425,3,63
4,SR-ARE,multitask_gnn,847,0.205431,0.5,0.772967,0.492375,0.291302,0.565525,0.677593,...,0.450746,0.249033,0.325096,0.695137,0.103448,0.349010,328,345,23,151
5,SR-MMP,multitask_gnn,842,0.220903,0.5,0.818450,0.544056,0.364457,0.637767,0.719381,...,0.513557,0.230644,0.290690,0.664934,0.000000,0.399464,376,280,25,161


### Completion gate
Confirm that the declared artifacts exist before continuing to `15_hyperparameter_optimization.ipynb`.